# R4 — Bloom filter + manual thresholds

R4 แทน similarity ของชื่อ plaintext ด้วย Dice similarity บน Bloom-filter bigrams เพื่อดู privacy/accuracy trade-off.

In [ ]:
from pathlib import Path
import sys, json, inspect
import pandas as pd
from IPython.display import Markdown, display

def find_root():
    candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents,
                  Path(r'D:/66070260-Year3_Term2/Project1/Code')]
    for candidate in candidates:
        if (candidate / 'exp_lib.py').exists(): return candidate
    raise FileNotFoundError('Project root containing exp_lib.py was not found')

ROOT = find_root(); EXP = ROOT / 'experiments'
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))

def source(module, *names):
    for name in names:
        display(Markdown(f'### `{module.__name__}.{name}`'))
        print(inspect.getsource(getattr(module, name)))

def read_json(relative):
    return json.loads((ROOT / relative).read_text(encoding='utf-8'))

print('Project root:', ROOT)


## 1. Bigram → Bloom bit vector → Dice similarity

L เล็กทำให้ collision มากขึ้น: ปกปิดรายละเอียดมากขึ้น แต่อาจลดพลังแยกคู่.

In [ ]:
import exp_r4_bloom_privacy as r4
source(r4, 'bigrams', 'bloom_packed', 'dice_chunked', 'compute_pair_bloom_features')

## 2. Train probabilities และ threshold มือ

ทดลอง L = 2000, 1000, 500, 250 โดยใช้ threshold เดิมก่อน เพื่อ isolate ผลของ representation.

In [ ]:
source(r4, 'build_probabilities_for_L', 'eval_manual')
r = read_json('experiments/r4_privacy_tradeoff.json')
rows = [{'L': int(L), **x['R4_manual']['test']} for L, x in r['per_L'].items()]
display(pd.DataFrame(rows).sort_values('L', ascending=False))